# Step 2: Running the Evaluation Experiments
### Project: Evaluating the Impact of RAG on Reducing LLM Hallucinations

In this notebook, we run all 60 benchmark questions across 4 different setups:
1. **Baseline LLM (No RAG)**: Direct question without context
2. **RAG (Top-3 Strict)**: 3 retrieved chunks + strict refusal instruction
3. **RAG (Top-5 Strict)**: 5 retrieved chunks + strict refusal instruction
4. **RAG (Top-3 Loose)**: 3 retrieved chunks with permissive prompt (Ablation)

In [ ]:
import sys
import pandas as pd
from tqdm.auto import tqdm
sys.path.append('..')

from src.config import QUESTIONS_FILE, RESULTS_FILE, VECTOR_STORE_DIR, LLM_PROVIDER, LLM_MODEL
from src.data_loader import load_questions
from src.vector_store import SimpleVectorStore, EmbeddingEngine
from src.llm_client import LLMClient
from src.rag_pipeline import BaselinePipeline, RAGPipeline
from src.evaluator import Evaluator

print(f"Active Provider: {LLM_PROVIDER.upper()} | Model: {LLM_MODEL}")

## 1. Load Vector Store & Benchmark Questions
We inspect the 60 questions categorized across Direct Fact, Multi-Hop, Out-of-Corpus, and Adversarial.

In [ ]:
vector_store = SimpleVectorStore.load(VECTOR_STORE_DIR)
questions = load_questions(QUESTIONS_FILE)
print(f"Loaded {len(questions)} evaluation questions.")
df_q = pd.DataFrame(questions)
df_q['category'].value_counts()

## 2. Initialize Pipelines & Evaluator

In [ ]:
llm = LLMClient()
baseline_pipe = BaselinePipeline(llm_client=llm)
rag_k3_strict = RAGPipeline(vector_store=vector_store, llm_client=llm, top_k=3, strict_grounding=True)
rag_k5_strict = RAGPipeline(vector_store=vector_store, llm_client=llm, top_k=5, strict_grounding=True)
rag_k3_loose = RAGPipeline(vector_store=vector_store, llm_client=llm, top_k=3, strict_grounding=False)
evaluator = Evaluator(llm_client=llm)
print("All 4 pipelines initialized successfully.")

## 3. Run the Benchmark Loop across All Questions

In [ ]:
results_data = []

for row in tqdm(questions, desc="Evaluating Questions"):
    q_id = row["id"]
    category = row["category"]
    q_text = row["question"]
    gt = row["ground_truth"]
    in_corpus = row["in_corpus"].strip().lower() == "true"
    
    # Run Baseline
    r_base = baseline_pipe.query(q_text)
    e_base = evaluator.evaluate_sample(q_text, r_base["answer"], gt, category, in_corpus)
    
    # Run RAG Top-3 Strict
    r_rag3 = rag_k3_strict.query(q_text)
    e_rag3 = evaluator.evaluate_sample(q_text, r_rag3["answer"], gt, category, in_corpus, context=r_rag3["retrieved_context"])
    
    # Run RAG Top-5 Strict
    r_rag5 = rag_k5_strict.query(q_text)
    e_rag5 = evaluator.evaluate_sample(q_text, r_rag5["answer"], gt, category, in_corpus, context=r_rag5["retrieved_context"])
    
    # Run RAG Top-3 Loose
    r_ragl = rag_k3_loose.query(q_text)
    e_ragl = evaluator.evaluate_sample(q_text, r_ragl["answer"], gt, category, in_corpus, context=r_ragl["retrieved_context"])
    
    results_data.append({
        "id": q_id,
        "category": category,
        "question": q_text,
        "ground_truth": gt,
        "in_corpus": in_corpus,
        "baseline_answer": r_base["answer"],
        "baseline_hallucinated": e_base["hallucinated"],
        "baseline_faithfulness": e_base["faithfulness"],
        "rag_k3_answer": r_rag3["answer"],
        "rag_k3_hallucinated": e_rag3["hallucinated"],
        "rag_k3_faithfulness": e_rag3["faithfulness"],
        "rag_k5_answer": r_rag5["answer"],
        "rag_k5_hallucinated": e_rag5["hallucinated"],
        "rag_loose_answer": r_ragl["answer"],
        "rag_loose_hallucinated": e_ragl["hallucinated"]
    })

df_res = pd.DataFrame(results_data)
df_res.to_csv(RESULTS_FILE, index=False)
print(f"Experiment execution complete! Saved {len(df_res)} rows to {RESULTS_FILE}")